# Clean and export the KíLín repository

This notebook prepares a safe source snapshot of KíLín for review and
collaboration. It has two separate operations:

1. **Preview and optionally remove** recognized, disposable cache files and
   operating-system metadata. In the supplied configuration, deletion is **enabled** (`DELETE = True`).
2. **Select, preview, and optionally export** chosen top-level files and
   folders. The default selection includes the required `LICENSE`,
   `CITATION.cff`, `README.pypi.md`, `MANIFEST.in`, Git-tracked
   `.github/` workflows, `.gitignore`, an empty `sandbox/` directory,
   and `_si_article_/`, which includes the Git-tracked benchmark sources,
   empty benchmark workspaces, empty repository placeholders, the paper's
   `code_analysis/` sources, and the reference-analysis placeholder.

Edit `ZIP_TOP_LEVEL_SELECTION` to choose the top-level entries in the ZIP.
Only entries explicitly enabled in that mapping can appear in the archive.
The contents of selected source folders come from Git-tracked working-tree
files, so uncommitted edits to tracked files are included. Untracked and
ignored files are not added automatically; add new source files to Git first.

Run the notebook from the KíLín checkout (normally in `assets/`). Git is
required for export, but not for the limited cleanup.

**Protected data:** Neither operation deletes or exports the contents of
`sandbox/`, `_si_article_/code_repo/`, `_si_article_/literature_repo/`,
local model caches, virtual environments, `.git/`, or benchmark workspace
contents. Git-tracked benchmark and code-analysis sources are exported.
The exporter does **not** clean the working tree.


In [ ]:
from __future__ import annotations

import os
import shutil
import stat
import subprocess
import tempfile
import tomllib
import zipfile
from pathlib import Path


In [ ]:
def find_repository_root(start: Path | None = None) -> Path:
    """Find the KíLín checkout containing the expected project markers.

    Parameters
    ----------
    start : Path | None
        Starting location, or the current working directory.

    Returns
    -------
    Path
        Resolved KíLín repository root.

    Raises
    ------
    RuntimeError
        If the location is not inside a recognizable KíLín checkout.

    """

    current = (start or Path.cwd()).resolve()
    if current.is_file():
        current = current.parent

    for directory in (current, *current.parents):
        pyproject = directory / "pyproject.toml"
        if not (
            pyproject.is_file()
            and (directory / "kilin" / "__init__.py").is_file()
            and (directory / "assets" / "clean_repository.ipynb").is_file()
        ):
            continue

        try:
            project = tomllib.loads(
                pyproject.read_text(encoding="utf-8")
            )
        except (OSError, ValueError) as error:
            raise RuntimeError(
                f"Invalid project metadata in {pyproject}."
            ) from error

        if project.get("project", {}).get("name") == "kilin":
            return directory

    raise RuntimeError(
        "Could not find the KíLín repository root from "
        f"{current}. Open this notebook within the source checkout."
    )


In [ ]:
repository_root = find_repository_root()
print(f"KíLín repository: {repository_root}")


## Cleanup policy

Only explicitly recognized disposable artifacts are candidates for removal.
The scan never enters user data, model caches, Git internals, virtual
environments, benchmark workspaces, or local code/literature repositories. Tracked paths are never deleted, even
if they happen to match a temporary-file rule. AppleDouble `._*` files must
have the AppleDouble signature, to avoid deleting unrelated hidden files.

The exclusions in `.gitignore` govern what Git tracks; **they are not a
list of files that the cleaner is allowed to delete**. In particular, ignored
model weights and workspaces are always protected.


In [ ]:
TEMPORARY_DIRECTORY_NAMES = {
    "__pycache__",
    "__MACOSX",
    ".ipynb_checkpoints",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".coverage_cache",
}

TEMPORARY_FILE_NAMES = {
    ".DS_Store",
    ".coverage",
    "Thumbs.db",
}

TEMPORARY_FILE_SUFFIXES = {".pyc", ".pyo"}

PROTECTED_DIRECTORY_NAMES = {
    ".git",
    ".venv",
    "venv",
    "env",
    ".models",
    ".weights",
    "sandbox",
    ".tox",
    ".nox",
}

PROTECTED_RELATIVE_DIRECTORIES = {
    ("_si_article_", "benchmarks", "sandbox"),
    ("_si_article_", "benchmarks", "results"),
    ("_si_article_", "code_repo"),
    ("_si_article_", "literature_repo"),
}

APPLEDOUBLE_MAGIC = bytes.fromhex("00051607")


In [ ]:
def git_tracked_paths(root: Path) -> set[str] | None:
    """Return Git-tracked paths, or None outside a Git checkout.

    Paths are repository-relative POSIX strings. An invalid Git checkout
    raises an error instead of allowing an unsafe fallback.

    Parameters
    ----------
    root : Path
        Resolved KíLín repository root.

    Returns
    -------
    set[str] | None
        Tracked relative paths, or None when Git metadata is absent.

    """

    if not (root / ".git").exists():
        return None

    try:
        result = subprocess.run(
            ["git", "-C", str(root), "rev-parse", "--show-toplevel"],
            capture_output=True,
            check=True,
            text=True,
        )
        git_root = Path(result.stdout.strip()).resolve()
        if git_root != root.resolve():
            raise RuntimeError(
                f"Git root {git_root} differs from KíLín root {root}."
            )

        files = subprocess.run(
            ["git", "-C", str(root), "ls-files", "--cached", "-z"],
            capture_output=True,
            check=True,
        )
    except (OSError, subprocess.CalledProcessError) as error:
        raise RuntimeError("Could not read the Git file inventory.") from error

    return {
        os.fsdecode(item).replace(os.sep, "/")
        for item in files.stdout.split(b"\x00")
        if item
    }


def is_protected(path: Path, root: Path) -> bool:
    """Return whether a path belongs to protected repository data.

    Parameters
    ----------
    path : Path
        Candidate path.
    root : Path
        Resolved repository root.

    Returns
    -------
    bool
        True when any component or protected prefix must be preserved.

    """

    parts = path.relative_to(root).parts
    return (
        any(part in PROTECTED_DIRECTORY_NAMES for part in parts)
        or any(
            parts[:len(prefix)] == prefix
            for prefix in PROTECTED_RELATIVE_DIRECTORIES
        )
    )


def is_appledouble(path: Path) -> bool:
    """Identify a real AppleDouble companion file by its header.

    Parameters
    ----------
    path : Path
        Candidate file beginning with ``._``.

    Returns
    -------
    bool
        Whether it contains the AppleDouble file signature.

    """

    if not path.name.startswith("._") or path.is_symlink():
        return False
    try:
        with path.open("rb") as file:
            return file.read(4) == APPLEDOUBLE_MAGIC
    except OSError:
        return False


## Preview removable paths

The scanner avoids following symbolic links and skips directories under
protected locations. Candidates tracked by Git are excluded. Review these
lists before running the enabled deletion step (`DELETE = True`).


In [ ]:
def find_temporary_paths(
    root: Path,
) -> tuple[list[Path], list[Path]]:
    """Identify safe-to-remove generated paths below the checkout.

    Parameters
    ----------
    root : Path
        Resolved KíLín repository root.

    Returns
    -------
    tuple[list[Path], list[Path]]
        Candidate directories and files, sorted by relative path.

    """

    tracked = git_tracked_paths(root) or set()
    directories: list[Path] = []
    files: list[Path] = []

    for parent, names, filenames in os.walk(
        root, topdown=True, followlinks=False
    ):
        directory = Path(parent)
        retained_dirs = []

        for name in sorted(names):
            path = directory / name
            if path.is_symlink() or is_protected(path, root):
                continue

            relative = path.relative_to(root).as_posix()
            contains_tracked = any(
                item == relative or item.startswith(relative + "/")
                for item in tracked
            )
            if name in TEMPORARY_DIRECTORY_NAMES:
                if not contains_tracked:
                    directories.append(path)
                continue

            retained_dirs.append(name)

        names[:] = retained_dirs
        for name in sorted(filenames):
            path = directory / name
            if path.is_symlink() or is_protected(path, root):
                continue
            if path.relative_to(root).as_posix() in tracked:
                continue
            if (
                name in TEMPORARY_FILE_NAMES
                or path.suffix in TEMPORARY_FILE_SUFFIXES
                or is_appledouble(path)
            ):
                files.append(path)

    directories.sort(key=lambda path: path.relative_to(root).as_posix())
    files.sort(key=lambda path: path.relative_to(root).as_posix())
    return directories, files


In [ ]:
directories, files = find_temporary_paths(repository_root)

print("Removable directories:")
for path in directories:
    print("  ", path.relative_to(repository_root))
print("Removable files:")
for path in files:
    print("  ", path.relative_to(repository_root))
print(f"Preview: {len(directories)} directories, {len(files)} files.")


## Optional removal

**Deletion is enabled** (`DELETE = True`). Inspect the candidate
lists before running the removal cell. Set `DELETE = False` to preview only. A fresh scan is performed immediately before deletion;
protected and Git-tracked paths remain excluded. No export operation requires
running the deletion step.


In [ ]:
DELETE = True  # Enabled; inspect the preview before running cleanup.


In [ ]:
if not DELETE:
    print("No files deleted. Set DELETE = True to remove candidates.")
else:
    # Re-scan instead of trusting an earlier, potentially stale preview.
    directories, files = find_temporary_paths(repository_root)
    removed_directories = 0
    removed_files = 0

    for path in files:
        if (
            not path.is_symlink()
            and path.is_file()
            and path.resolve().is_relative_to(repository_root)
        ):
            path.unlink()
            removed_files += 1

    for path in directories:
        if (
            not path.is_symlink()
            and path.is_dir()
            and path.resolve().is_relative_to(repository_root)
        ):
            shutil.rmtree(path)
            removed_directories += 1

    print(
        f"Removed {removed_directories} directories "
        f"and {removed_files} files."
    )


In [ ]:
remaining_directories, remaining_files = find_temporary_paths(
    repository_root
)
if remaining_directories or remaining_files:
    print(
        f"Remaining candidates: {len(remaining_directories)} directories, "
        f"{len(remaining_files)} files."
    )
else:
    print("No removable temporary paths remain.")


## Select top-level entries for the source ZIP

Change the Boolean values in `ZIP_TOP_LEVEL_SELECTION` below to choose
which files or folders appear in the ZIP. The default selection includes
repository sources, the mandatory `LICENSE`, `CITATION.cff`,
`README.pypi.md`, `MANIFEST.in`, Git-tracked `.github/` workflows,
`.gitignore`, `_si_article_/`, and an empty `sandbox/` directory.

- Names ending in `/` denote top-level folders; other names denote files.
- You may add another top-level name explicitly if needed later. It must be
  present in the checkout, and its contents must satisfy the safety policy.
- `.github/` is selected by default. If any required workflow is missing or
  untracked, export fails instead of silently omitting CI configuration.
- Selected source folders contain only Git-tracked working-tree files.
  Untracked files are reported (except protected contents) but not included.
- `LICENSE` is required for every source ZIP. It must exist and be
  Git-tracked; export fails if it is missing, untracked, or disabled.
- `CITATION.cff` must exist and be Git-tracked when selected. A missing
  or untracked citation file stops export instead of being silently omitted.
- `.gitignore` may be included even if it is not yet tracked by Git.
- `sandbox/` is exported as an empty directory, regardless of local contents.
- `_si_article_/benchmarks/` retains the existing benchmark policy: tracked
  `common.py`, `gold/`, and `notebooks/` are exported, while its `sandbox/`
  and `results/` directories remain empty in the ZIP.
- `_si_article_/code_repo/` and `_si_article_/literature_repo/` are created
  in the ZIP but **no contents are exported**, including tracked files.
  The cleanup scanner also protects these local repositories from deletion.
- `_si_article_/code_analysis/` exports Git-tracked paper-analysis sources.
  Move the existing `paper/` content there and add it to Git to include it;
  this source snapshot does not contain the original paper folder.
- `_si_article_/reference_analysis/` contains only `.gitkeep` for now.
  Other files are excluded until the export policy is deliberately changed.
- Git metadata, virtual environments, local model caches, generated artifacts,
  and operating-system metadata remain excluded even when selected.

The ZIP is written outside the repository. Existing ZIPs are never
replaced. The exporter does not delete or edit any repository files.


In [ ]:
EXPORT_SKIPPED_DIRECTORY_NAMES = (
    PROTECTED_DIRECTORY_NAMES | TEMPORARY_DIRECTORY_NAMES
)

EXPORT_SKIPPED_FILE_NAMES = TEMPORARY_FILE_NAMES | {
    "Thumbs.db",
}


SI_ARTICLE_REQUIRED_DIRECTORIES = (
    "_si_article_/benchmarks/",
    "_si_article_/benchmarks/sandbox/",
    "_si_article_/benchmarks/results/",
    "_si_article_/code_repo/",
    "_si_article_/literature_repo/",
    "_si_article_/code_analysis/",
    "_si_article_/reference_analysis/",
)

SI_ARTICLE_EMPTY_DIRECTORIES = (
    "sandbox/",
    "_si_article_/benchmarks/sandbox/",
    "_si_article_/benchmarks/results/",
    "_si_article_/code_repo/",
    "_si_article_/literature_repo/",
)

SI_ARTICLE_REFERENCE_PLACEHOLDER = (
    "_si_article_/reference_analysis/.gitkeep"
)


REQUIRED_CI_WORKFLOWS = frozenset({
    ".github/workflows/tests.yml",
    ".github/workflows/package.yml",
    ".github/workflows/docs.yml",
})


def should_export(relative: str) -> bool:
    """Apply explicit safety exclusions to a tracked source path.

    Parameters
    ----------
    relative : str
        Repository-relative path using POSIX separators.

    Returns
    -------
    bool
        Whether the path is safe to include in the exported source ZIP.

    """

    parts = tuple(relative.split("/"))
    if not parts or relative.startswith("/") or ".." in parts:
        return False
    if any(part in EXPORT_SKIPPED_DIRECTORY_NAMES for part in parts):
        return False
    if any(
        parts[:len(prefix)] == prefix
        for prefix in PROTECTED_RELATIVE_DIRECTORIES
    ):
        return False
    if (
        relative.startswith("_si_article_/reference_analysis/")
        and relative != SI_ARTICLE_REFERENCE_PLACEHOLDER
    ):
        return False
    filename = parts[-1]
    if filename.startswith("._"):
        return False
    if filename in EXPORT_SKIPPED_FILE_NAMES:
        return False
    if filename in {".env", ".coverage"}:
        return False
    if filename.startswith(".env.") and filename != ".env.example":
        return False
    if Path(filename).suffix in TEMPORARY_FILE_SUFFIXES:
        return False
    return True


def resolve_top_level_selection(
    root: Path,
    selection: dict[str, bool],
) -> tuple[set[str], set[str]]:
    """Validate the requested top-level files and directories.

    A selected sandbox is represented only by an empty directory entry.
    The selected SI article includes protected empty repository and benchmark
    workspace directories. All selected paths must be within the checkout.

    Parameters
    ----------
    root : Path
        Resolved KíLín repository root.
    selection : dict[str, bool]
        Top-level file names and folder names (ending in ``/``) mapped to
        Boolean inclusion choices.

    Returns
    -------
    tuple[set[str], set[str]]
        Selected file names and selected directory entries.

    Raises
    ------
    RuntimeError
        If the licence is not selected, or a selected entry is
        invalid, missing, or protected.

    """

    root = root.resolve()
    if not selection:
        raise RuntimeError("Select at least one top-level ZIP entry.")
    if selection.get("LICENSE") is not True:
        raise RuntimeError("LICENSE must be included in every source ZIP.")

    files: set[str] = set()
    directories: set[str] = set()
    for name, enabled in selection.items():
        if not isinstance(name, str) or not isinstance(enabled, bool):
            raise RuntimeError("ZIP choices must be names and Booleans.")
        is_directory = name.endswith("/")
        base = name[:-1] if is_directory else name
        if (
            not base
            or base in {".", ".."}
            or "/" in base
            or "\\" in base
        ):
            raise RuntimeError(f"Invalid top-level ZIP entry: {name!r}.")
        if not enabled:
            continue

        if base == "sandbox":
            if not is_directory:
                raise RuntimeError("Select sandbox as 'sandbox/'.")
            directories.add("sandbox/")
            continue
        if not should_export(base):
            raise RuntimeError(f"Protected top-level entry: {name!r}.")

        source = root / base
        if (
            source.is_symlink()
            or not source.resolve().is_relative_to(root)
        ):
            raise RuntimeError(f"Unsafe top-level ZIP entry: {name!r}.")
        if is_directory:
            if not source.is_dir():
                raise RuntimeError(f"Missing source folder: {name!r}.")
            directories.add(name)
            if base == "_si_article_":
                for relative in SI_ARTICLE_REQUIRED_DIRECTORIES:
                    child = root / relative
                    if (
                        child.is_symlink()
                        or not child.is_dir()
                        or not child.resolve().is_relative_to(root)
                    ):
                        raise RuntimeError(
                            f"Missing or unsafe SI article folder: "
                            f"{relative!r}."
                        )
                directories.update(SI_ARTICLE_REQUIRED_DIRECTORIES)
        else:
            if not source.is_file():
                raise RuntimeError(f"Missing source file: {name!r}.")
            files.add(name)

    if not files and not directories:
        raise RuntimeError("Enable at least one top-level ZIP entry.")
    if {name + "/" for name in files} & directories:
        raise RuntimeError("Choose a file or folder, not both.")
    return files, directories


def list_source_files(
    root: Path,
    selection: dict[str, bool],
) -> tuple[list[str], list[str]]:
    """List Git-tracked inputs for the selected top-level entries.

    Selected top-level files must be tracked, except ``.gitignore``.
    Selected folders may be empty, but their file contents must be
    Git-tracked and pass the existing export safety policy.

    Parameters
    ----------
    root : Path
        Resolved KíLín checkout root.
    selection : dict[str, bool]
        Explicit choices of top-level files and directories.

    Returns
    -------
    tuple[list[str], list[str]]
        Source paths and omitted untracked paths in selected areas.

    Raises
    ------
    RuntimeError
        If Git is unavailable or any selected source path is unsafe.

    """

    root = root.resolve()
    selected_files, selected_dirs = resolve_top_level_selection(
        root, selection
    )
    tracked = git_tracked_paths(root)
    if tracked is None:
        raise RuntimeError(
            "ZIP export requires a Git checkout; .git is missing."
        )

    untracked_files = selected_files - tracked - {".gitignore"}
    if untracked_files:
        raise RuntimeError(
            "Selected root files are not Git-tracked: "
            + ", ".join(sorted(untracked_files))
        )

    def is_selected(relative: str) -> bool:
        """Check that a file belongs to an enabled top-level entry."""

        return (
            relative in selected_files
            or relative.split("/", 1)[0] + "/" in selected_dirs
        )

    selected = {
        item for item in tracked
        if should_export(item) and is_selected(item)
    }
    if ".gitignore" in selected_files:
        selected.add(".gitignore")
    if ".github/" in selected_dirs:
        missing = sorted(REQUIRED_CI_WORKFLOWS - selected)
        if missing:
            raise RuntimeError(
                "Missing Git-tracked CI workflows: " + ", ".join(missing)
            )
    if (
        "_si_article_/" in selected_dirs
        and SI_ARTICLE_REFERENCE_PLACEHOLDER not in selected
    ):
        raise RuntimeError(
            "The reference-analysis .gitkeep must be Git-tracked."
        )
    for relative in selected:
        source = root / relative
        if (
            not source.is_file()
            or source.is_symlink()
            or not source.resolve().is_relative_to(root)
        ):
            raise RuntimeError(
                f"Missing, symbolic, or unsafe tracked file: {relative}"
            )

    try:
        untracked_output = subprocess.run(
            ["git", "-C", str(root), "ls-files", "--others",
             "--exclude-standard", "-z"],
            capture_output=True,
            check=True,
        ).stdout
    except (OSError, subprocess.CalledProcessError) as error:
        raise RuntimeError("Could not inspect untracked files.") from error

    untracked = sorted(
        relative
        for item in untracked_output.split(b"\x00")
        if item
        for relative in [os.fsdecode(item).replace(os.sep, "/")]
        if (
            relative != ".gitignore"
            and is_selected(relative)
            and should_export(relative)
        )
    )
    return sorted(selected), untracked


In [ ]:
# Choose the top-level entries for the archive. Change True to False
# to omit an entry, except LICENSE, which must remain True.
# No other top-level entries are included by default.
ZIP_TOP_LEVEL_SELECTION = {
    "README.md": True,
    "README.pypi.md": True,
    "LICENSE": True,
    "CITATION.cff": True,
    "MANIFEST.in": True,
    "pyproject.toml": True,
    "uv.lock": True,
    ".gitignore": True,
    ".github/": True,  # Export Git-tracked CI workflows.
    "assets/": True,
    "_si_article_/": True,  # Benchmarks, analyses, empty repositories.
    "docs/": True,
    "kilin/": True,
    "setup/": True,
    "tests/": True,
    "tutorials/": True,
    "sandbox/": True,  # Included as an empty folder only.
}


In [ ]:
selected_files, selected_directories = resolve_top_level_selection(
    repository_root, ZIP_TOP_LEVEL_SELECTION
)
source_files, untracked_files = list_source_files(
    repository_root, ZIP_TOP_LEVEL_SELECTION
)

print("ZIP top-level selection:")
for name, enabled in ZIP_TOP_LEVEL_SELECTION.items():
    print(f"  [{'x' if enabled else ' '}] {name}")
print(f"ZIP preview: {len(source_files)} source files.")
if "sandbox/" in selected_directories:
    print("The sandbox/ directory will be empty in the ZIP.")
for name in SI_ARTICLE_EMPTY_DIRECTORIES:
    if name in selected_directories:
        print(f"The {name} directory will be empty in the ZIP.")
if "_si_article_/" in selected_directories:
    print("code_analysis/ includes Git-tracked files only.")
    print("reference_analysis/ includes only .gitkeep.")
if untracked_files:
    print(
        f"WARNING: {len(untracked_files)} untracked files in selected "
        "areas will be omitted:"
    )
    for item in untracked_files:
        print("  ", item)
print("Verify that intended new source files are tracked before export.")


In [ ]:
def export_source_zip(
    root: Path,
    destination: Path,
    source_files: list[str],
    selection: dict[str, bool],
) -> Path:
    """Write and verify an archive limited to selected top-level entries.

    The archive is created in a temporary file in the destination folder.
    Files are streamed from the working tree, and every directory entry has
    a fixed timestamp for reproducible exports.

    Parameters
    ----------
    root : Path
        Resolved KíLín checkout root.
    destination : Path
        Destination ZIP path, outside the checkout.
    source_files : list[str]
        Validated relative file paths from :func:`list_source_files`.
    selection : dict[str, bool]
        Enabled top-level files and folders to export.

    Returns
    -------
    Path
        Path to the verified ZIP file.

    Raises
    ------
    RuntimeError
        If the ZIP would include an unsafe source or fails verification.
    FileExistsError
        If the requested archive already exists.

    """

    root = root.resolve()
    destination = destination.resolve()
    if destination.is_relative_to(root):
        raise RuntimeError("Export the archive outside the checkout.")
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite {destination}.")
    if not destination.parent.is_dir():
        raise RuntimeError("The ZIP destination directory does not exist.")

    top_files, top_directories = resolve_top_level_selection(
        root, selection
    )
    expected = set(source_files)
    if not top_files.issubset(expected):
        raise RuntimeError("A selected top-level source file is missing.")
    if not all(
        should_export(item)
        and (
            item in top_files
            or item.split("/", 1)[0] + "/" in top_directories
        )
        for item in expected
    ):
        raise RuntimeError("The ZIP includes an unselected or unsafe path.")
    empty_directories = SI_ARTICLE_EMPTY_DIRECTORIES
    if any(
        item.startswith(prefix) for item in expected
        for prefix in empty_directories
    ):
        raise RuntimeError("Protected workspace directories must be empty.")

    directories = set(top_directories)
    for relative in expected:
        parent = Path(relative).parent
        while parent != Path("."):
            directories.add(parent.as_posix() + "/")
            parent = parent.parent

    timestamp = (1980, 1, 1, 0, 0, 0)
    with tempfile.NamedTemporaryFile(
        prefix=".kilin-export-",
        suffix=".zip.tmp",
        dir=destination.parent,
        delete=False,
    ) as temporary:
        temp_path = Path(temporary.name)

    try:
        with zipfile.ZipFile(temp_path, "w") as archive:
            for relative in sorted(directories):
                info = zipfile.ZipInfo(relative, timestamp)
                info.create_system = 3
                info.external_attr = (stat.S_IFDIR | 0o755) << 16
                info.external_attr |= 0x10
                archive.writestr(info, b"")

            for relative in sorted(expected):
                source = root / relative
                if (
                    not source.is_file()
                    or source.is_symlink()
                    or not source.resolve().is_relative_to(root)
                ):
                    raise RuntimeError(f"Unsafe source: {relative}")
                info = zipfile.ZipInfo(relative, timestamp)
                info.create_system = 3
                info.external_attr = (
                    stat.S_IFREG | (source.stat().st_mode & 0o777)
                ) << 16
                info.compress_type = zipfile.ZIP_DEFLATED
                with source.open("rb") as input_file:
                    with archive.open(info, "w") as output_file:
                        shutil.copyfileobj(input_file, output_file)

        with zipfile.ZipFile(temp_path) as archive:
            names = archive.namelist()
            if (
                len(names) != len(set(names))
                or set(names) != expected | directories
                or archive.testzip() is not None
                or any(
                    name.startswith(prefix) and name != prefix
                    for name in names for prefix in empty_directories
                )
            ):
                raise RuntimeError("ZIP integrity or content check failed.")

        temp_path.replace(destination)
    finally:
        temp_path.unlink(missing_ok=True)

    return destination


## Optional ZIP creation

The destination defaults to the **parent of the repository**, so it cannot
be accidentally included in the source tree. Change the filename if an
archive already exists: overwriting an existing ZIP is intentionally refused.

ZIP creation is enabled (`EXPORT_ZIP = True`). Set it to `False` to
preview without exporting.
You do not need to turn on cleanup to export a clean archive.


In [ ]:
EXPORT_ZIP = True  # Enabled; inspect the inputs before exporting.
ZIP_PATH = repository_root.parent / (
    f"{repository_root.name}_source.zip"
)


In [ ]:
if not EXPORT_ZIP:
    print(f"ZIP preview only. Set EXPORT_ZIP = True for {ZIP_PATH}.")
else:
    # Recompute selected inputs so no stale preview is exported.
    source_files, untracked_files = list_source_files(
        repository_root, ZIP_TOP_LEVEL_SELECTION
    )
    if untracked_files:
        print(
            f"NOTE: {len(untracked_files)} untracked source files "
            "in selected areas are omitted."
        )
    archive_path = export_source_zip(
        repository_root,
        ZIP_PATH,
        source_files,
        ZIP_TOP_LEVEL_SELECTION,
    )
    print(f"Verified clean source archive: {archive_path}")
